# 4 · Type extension  (growing vocabulary of minted types)

Turns one-off minted types into a **persistent, growing vocabulary** and matches future articles
against **schema.org ∪ vocabulary**. Recipe (after EDC self-canonicalization and Feng/Wu/Meng):
*define → embed → retrieve nearest existing terms → LLM-verify → mint only on no-match.*

- **Hook A** — retrieval offers vocabulary types as candidates alongside schema.org (reuse by matching).
- **Hook B** — when the LLM mints, canonicalize the mint against the vocabulary (embed + LLM-verify);
  merge onto an existing concept or admit a new entry (adding the new label as an alias = its synonyms).

**Self-contained**: nb4 does its own matching, type-selection, **and** within-group dedup — so it needs
only **nb1** (pools) and **nb2** (`oie_entities.json`); **nb3 is optional** (run it only for the
schema.org-only baseline you want to compare against). Writes `results/type_vocabulary.json` and
per-article `typed_deduped_entities_ext.json`. `EXPAND_MODE` toggles grow vs. closed-schema.

In [52]:
# --- Config: type-extension stage --------------------------------------------------------
import json, re, os, copy
from pathlib import Path
from collections import Counter
import numpy as np

SCHEMA_DIR  = Path("schema_org_files")     # pools from nb1
RESULTS_DIR = Path("results")              # reads oie_entities.json (nb2); self-contained, nb3 not required
VOCAB_PATH  = RESULTS_DIR / "type_vocabulary.json"

MATCH_MODEL  = "intfloat/e5-base-v2"       # ONE embedding model for the whole vocabulary (don't mix spaces)
GEMINI_MODEL = "gemini-2.5-flash"
EXPAND_MODE  = True                        # True: admit minted types (grow). False: restrict to schema.org u vocab.

# Hook B (mint canonicalization) call budget — matters for the Gemini daily-quota:
#   "per_mint"    : one verify call PER minted type. More calls, but same-article sibling mints can
#                   merge into each other (the vocab grows between mints within the article).
#   "per_article" : ONE batched verify call for ALL of an article's mints at once — far fewer API calls.
#                   Trade-off: sibling mints in the same article can't merge with each other here; that
#                   only happens later via the global consolidate_vocab pass.
CANON_MODE   = "per_article"

TOP_K_MATCH  = 5                           # candidates shown to the type-selection LLM (Hook A)
TOP_K_CANON  = 5                           # nearest vocab entries shown to the canonicalization verifier (Hook B)
ALIGN_QUERY_STYLE = True                   # append "Also called <synonyms>." to each OIE type query (as in nb3)

# Which articles nb4 runs on — and the order the vocabulary grows along (order matters: incremental).
#   None  -> every article that has an oie_entities.json (from nb2), sorted.
#   list  -> exactly these ids, in this order, e.g. ["art2", "art1", "art20"].
#SELECTED_ARTICLES = None
SELECTED_ARTICLES = ["art1", "art2", "art4", "art5"]

assert CANON_MODE in ("per_mint", "per_article"), "CANON_MODE must be 'per_mint' or 'per_article'"
_available = sorted(p.parent.name for p in RESULTS_DIR.glob("*/oie_entities.json"))
if SELECTED_ARTICLES is None:
    ARTICLE_ORDER = _available
else:
    missing = [a for a in SELECTED_ARTICLES if a not in _available]
    if missing:
        print("WARNING: skipping (no oie_entities.json):", missing)
    ARTICLE_ORDER = [a for a in SELECTED_ARTICLES if a in _available]
assert ARTICLE_ORDER, "No runnable articles selected (need nb2 oie_entities.json). Check SELECTED_ARTICLES."
print("expand:", EXPAND_MODE, "| canon:", CANON_MODE, "| match:", MATCH_MODEL, "| articles:", ARTICLE_ORDER)

expand: True | canon: per_article | match: intfloat/e5-base-v2 | articles: ['art1', 'art2', 'art4', 'art5']


In [41]:
# First run only (sentence-transformers pulls torch; google-genai for the LLM stages).
%pip install -q sentence-transformers google-genai numpy

Note: you may need to restart the kernel to use updated packages.


In [42]:
# Embedding model (loaded once). e5 needs "query: " on the query side and "passage: " on documents;
# both the OIE-type queries and every document (schema.org OR vocab) go through the same convention.
from sentence_transformers import SentenceTransformer
_MODEL = SentenceTransformer(MATCH_MODEL)

def _prefixes(name):
    n = name.lower()
    if "e5" in n:  return "query: ", "passage: "
    if "bge" in n: return "Represent this sentence for searching relevant passages: ", ""
    return "", ""
QP, DP = _prefixes(MATCH_MODEL)

def embed(texts, prefix):
    return np.asarray(_MODEL.encode([prefix + t for t in texts],
                                    normalize_embeddings=True, batch_size=64))
print("embedder ready:", MATCH_MODEL, "| query_prefix", repr(QP), "| doc_prefix", repr(DP))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

embedder ready: intfloat/e5-base-v2 | query_prefix 'query: ' | doc_prefix 'passage: '


In [43]:
# Schema.org document pool (fixed) — embedded once. Vocabulary documents are embedded per article
# inside retrieval, since the vocabulary grows during the run.
schema_pool   = json.loads((SCHEMA_DIR / "schema_classes_enriched.json").read_text())
SCHEMA_LABELS = {c["label"] for c in schema_pool}
schema_emb    = embed([c["embedding_text"] for c in schema_pool], DP)
print(len(schema_pool), "schema.org classes embedded")

933 schema.org classes embedded


In [44]:
# The growing vocabulary store. One entry per canonical minted type:
#   id, canonical_label, definition, parent (schema.org label or vocab id), aliases[] (= its synonyms),
#   provenance[], support, first_seen.
def load_vocab():
    if VOCAB_PATH.exists():
        return json.loads(VOCAB_PATH.read_text())
    return {"types": [], "next_id": 1}

def save_vocab(vocab):
    VOCAB_PATH.write_text(json.dumps(vocab, indent=2, ensure_ascii=False))

def vocab_embedding_text(t):
    parent = t.get("parent") or "Thing"
    also   = f" Also called {', '.join(t['aliases'])}." if t.get("aliases") else ""
    return f'{t["canonical_label"]} ({parent}): {t.get("definition", "")}{also}'

def _find_vocab_by_label(vocab, label):
    for t in vocab["types"]:
        if t["canonical_label"] == label or label in t.get("aliases", []):
            return t
    return None

def _is_vocab_label(vocab, label):
    return _find_vocab_by_label(vocab, label) is not None

def _bump(ent, aid, oie_type, group):
    ent["support"] = ent.get("support", 0) + 1
    ent.setdefault("provenance", []).append(
        {"article": aid, "oie_type": oie_type, "mentions": [e["text"] for e in group]})

In [45]:
# Hook A — retrieve top-k candidates for each OIE type against schema.org u current vocabulary.
def retrieve_candidates(type_texts, vocab):
    q_emb = embed(type_texts, QP)
    meta  = [("schema.org", c) for c in schema_pool]
    d_emb = schema_emb
    if vocab["types"]:
        v_emb = embed([vocab_embedding_text(t) for t in vocab["types"]], DP)
        d_emb = np.vstack([schema_emb, v_emb])
        meta  = meta + [("vocab", t) for t in vocab["types"]]
    sims = q_emb @ d_emb.T
    topk = np.argsort(-sims, axis=1)[:, :TOP_K_MATCH]
    out = []
    for ti in range(len(type_texts)):
        row = []
        for j in topk[ti]:
            src, obj = meta[j]
            if src == "schema.org":
                row.append({"source": "schema.org", "label": obj["label"], "path": obj.get("path", ""),
                            "score": round(float(sims[ti, j]), 3),
                            "comment": obj.get("enriched_comment") or obj.get("comment", "")})
            else:
                row.append({"source": "vocab", "id": obj["id"], "label": obj["canonical_label"],
                            "path": obj.get("parent", ""), "score": round(float(sims[ti, j]), 3),
                            "comment": obj.get("definition", "")})
        out.append(row)
    return out

In [46]:
# Article inputs + Gemini client (JSON mode, temperature 0 for reproducibility).
def load_article(aid):
    oie   = json.loads((RESULTS_DIR / aid / "oie_entities.json").read_text())
    tinfo = {t["type"]: t for t in oie.get("types", [])}
    types = list(dict.fromkeys(e["type"] for e in oie["entities"]))
    def qtext(t):
        info = tinfo.get(t, {})
        q = f'{t}: {info.get("definition", "")}'.strip()
        if ALIGN_QUERY_STYLE and info.get("synonyms"):
            q = f'{q} Also called {", ".join(info["synonyms"])}.'
        return q
    groups = {}
    for e in oie["entities"]:
        groups.setdefault(e["type"], []).append(e)
    return oie, tinfo, types, [qtext(t) for t in types], groups

from google import genai
from google.genai import types as genai_types
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
assert GEMINI_API_KEY, "Set GEMINI_API_KEY (or GOOGLE_API_KEY) in the env and restart the kernel."
client = genai.Client(api_key=GEMINI_API_KEY)

def gemini_json(system, user):
    r = client.models.generate_content(
        model=GEMINI_MODEL, contents=user,
        config=genai_types.GenerateContentConfig(
            system_instruction=system, temperature=0, response_mime_type="application/json"))
    return r.text

def parse_json(text):
    s = (text or "").strip()
    if s.startswith("```"):
        s = re.sub(r"^```[a-zA-Z]*\s*", "", s); s = re.sub(r"\s*```$", "", s)
    return json.loads(s)

In [47]:
# ==== EDIT ME — prompts ==================================================================
# TYPE_DEDUP_PROMPT mirrors nb3's combined TASK 1 (type selection) + TASK 2 (within-group dedup),
# with two deltas: candidates may be Schema.org OR vocabulary (a "source" is reported), and the mint
# "parent" bullet lets a [vocab] candidate be the parent (nb3 is Schema.org-only). nb4 is
# self-contained (does its own dedup), so nb3 is NOT required.
TYPE_DEDUP_PROMPT = """You are an expert knowledge-engineering assistant. You are given several groups of entity mentions from one document. Each group shares a single free-form OIE type. For each group independently, perform two tasks, in order, and report them as separate fields (chosen_type / duplicate_clusters) — do not blend the two tasks, and NEVER merge or compare mentions across different groups.

Each group provides:
- OIE_TYPE: the free-form type label these mentions were given, plus its definition.
- CANDIDATES: a ranked list of candidate types proposed as matches for OIE_TYPE. Each candidate is either an existing Schema.org class or an existing type from our extended vocabulary; its source is marked [schema.org] or [vocab]. A candidate's `path` is its chain of ancestor classes, so every type you might use as a parent — a candidate label ([schema.org] or [vocab]) or any ancestor in a Schema.org candidate's path — is already listed here.
- MENTIONS: the verbatim surface strings that were all assigned OIE_TYPE.

TASK 1 — TYPE SELECTION  (output field: "chosen_type")
Choose the single best type for the group.
- Prefer an existing candidate (Schema.org or vocabulary). Pick the one whose meaning (judged from its comment and path) best fits OIE_TYPE and the mentions. A candidate that is somewhat broader or narrower but still preserves what makes the type distinctive is preferable to a new type.
- Instance test — apply before accepting any candidate, judging by the MENTIONS, not by the OIE_TYPE definition alone: would each mention individually be a valid instance of that candidate — i.e. would it sit naturally alongside the concrete examples listed in the candidate's comment? If the mentions are open-ended categories, roles, or kinds rather than specific instances of the candidate, the candidate is too broad; reject it.
- Mint a new type when no candidate passes the instance test — i.e. no candidate genuinely fits, including when the best candidate is only "correct" by being so broad that it loses the type's distinguishing meaning. Do not settle for an over-broad candidate; instead mint a concise PascalCase label and a one-sentence definition that captures the type's specific meaning.
- When you mint, also assign a parent: the closest existing type the minted type should be a subclass of, so it slots into the hierarchy. The parent may be a Schema.org class OR a vocabulary [vocab] candidate type — prefer the most specific suitable parent already shown for this group (a candidate label, whether [schema.org] or [vocab], or any ancestor in a Schema.org candidate's `path`). A broader [vocab] candidate that the minted type is a kind of makes an ideal parent. Use another real Schema.org class only if none of the candidates fit, and fall back to "Thing" as a last resort. Parent is required for minted types and null otherwise.
- Report: source ("schema.org" or "vocab" when you chose a candidate, matching that candidate's marked source; "new" when you mint), the chosen label (the exact candidate label, or the new PascalCase label), whether it is new, its definition and parent (both only when new; else null), and a one-line justification.

TASK 2 — WITHIN-GROUP DEDUPLICATION  (output field: "duplicate_clusters")
Cluster this group's mentions that refer to the same real-world entity. Only ever cluster mentions that belong to the same group.
- Be conservative. Merge two mentions ONLY when they clearly denote the same specific entity, such as a short form or abbreviation and its full form. When unsure, keep them separate. Under-merging is much better than over-merging.
- NEVER merge overlapping-but-distinct entities: mentions that overlap in surface form or reference but denote different real-world entities must stay separate.
- For each cluster pick the canonical mention: the most complete / most explicit surface form. The canonical and every alias must be exact strings copied from that group's mentions.
- Output only clusters of size >= 2. Do not list singletons.

Return ONLY valid JSON — no prose, no code fences. Output one result object per input group, in the same order, each echoing its exact oie_type string. Angle-bracket placeholders show the shape, not literal values:
{"results": [{"oie_type": "<oie type>", "chosen_type": {"source": "schema.org | vocab | new", "label": "<type label>", "is_new": false, "definition": null, "parent": null, "justification": "one short line"}, "duplicate_clusters": [{"canonical": "<canonical mention>", "aliases": ["<alias mention>"]}]}]}
Include every group exactly once. If a group has nothing to merge, use "duplicate_clusters": []."""

# Hook B verifier — used in CANON_MODE="per_mint" (one call per mint).
CANON_PROMPT = """You are canonicalizing a NEWLY MINTED entity type against an existing vocabulary, so that different labels for the same concept collapse onto one entry. You are given the new type (label + definition) and a short list of existing vocabulary types (id + label + definition).

Decide whether the new type denotes the SAME concept as one of the existing ones.
- Return the id of the matching existing type only if it is genuinely equivalent (same concept, allowing for different wording or label). Be conservative: do NOT match types that are merely related, overlapping, or broader/narrower — only true synonyms of the same concept.
- If none is equivalent, return null.

Return ONLY JSON: {"match_id": "<existing id or null>", "justification": "one short line"}"""

# Hook B verifier — used in CANON_MODE="per_article" (ONE call for all of an article's mints).
CANON_BATCH_PROMPT = """You are canonicalizing SEVERAL newly minted entity types against an existing vocabulary, so that different labels for the same concept collapse onto one entry. You are given several NEW TYPEs; each has its own key and its own short list of existing vocabulary CANDIDATES (id + label + definition).

For EACH new type independently, decide whether it denotes the SAME concept as one of ITS OWN listed candidates (never a candidate listed under a different new type).
- Return the id of the matching candidate only if it is genuinely equivalent (same concept, allowing for different wording or label). Be conservative: do NOT match types that are merely related, overlapping, or broader/narrower — only true synonyms of the same concept.
- If none of a new type's candidates is equivalent, return null for it.

Return ONLY JSON — one object per new type, echoing its exact key:
{"results": [{"key": "<key>", "match_id": "<candidate id or null>"}]}"""
print("prompts loaded")

prompts loaded


In [48]:
# Hook A caller — ONE Gemini call per article: type selection + within-group dedup (both tasks).
def llm_select_and_dedup(types, tinfo, groups, cands):
    blocks = []
    for i, t in enumerate(types):
        b = [f"GROUP {i+1} —", f'OIE_TYPE: "{t}"']
        if tinfo.get(t, {}).get("definition"):
            b.append(f'OIE_TYPE_DEFINITION: {tinfo[t]["definition"]}')
        b.append("CANDIDATES (ranked):")
        if cands[i]:
            for k, c in enumerate(cands[i], 1):
                path = c.get("path") or "(datatype)"
                b.append(f'  {k}. [{c["source"]}] {c["label"]} | path/parent: {path}')
                b.append(f'     {c.get("comment", "")}')
        else:
            b.append("  (none)")
        b.append("MENTIONS:")
        b += [f'  - "{e["text"]}"' for e in groups[t]]
        blocks.append("\n".join(b))
    user = f"You are given {len(types)} GROUPS from one document.\n\n" + "\n\n".join(blocks)
    parsed  = parse_json(gemini_json(TYPE_DEDUP_PROMPT, user))
    results = parsed.get("results", parsed) if isinstance(parsed, dict) else parsed
    out = {r.get("oie_type"): {"chosen_type": r.get("chosen_type") or {},
                               "clusters": r.get("duplicate_clusters") or []}
           for r in (results or []) if r.get("oie_type") in set(types)}
    for i, t in enumerate(types):                    # fallback: omitted group -> top candidate, no dedup
        if t not in out:
            top = cands[i][0] if cands[i] else {"source": "schema.org", "label": "Thing"}
            out[t] = {"chosen_type": {"source": top["source"], "label": top["label"], "is_new": False,
                                      "justification": "fallback: top candidate"}, "clusters": []}
    return out

# ---- Hook B — canonicalize a mint against the vocabulary --------------------------------------
# Shared step: nearest existing vocab entries to a mint (embeddings only, no LLM call).
def _mint_shortlist(label, definition, vocab):
    if not vocab["types"]:
        return []
    q     = embed([f"{label}: {definition}"], DP)
    v_emb = embed([vocab_embedding_text(t) for t in vocab["types"]], DP)
    order = np.argsort(-(q @ v_emb.T)[0])[:TOP_K_CANON]
    return [vocab["types"][j] for j in order]

# Verify ONE mint (CANON_MODE="per_mint"): returns the matching vocab entry or None.
def llm_canonicalize(label, definition, shortlist):
    if not shortlist:
        return None
    lines = ["NEW TYPE:", f"  label: {label}", f"  definition: {definition}", "", "EXISTING VOCABULARY TYPES:"]
    for t in shortlist:
        lines.append(f'  - id: {t["id"]} | label: {t["canonical_label"]} | definition: {t.get("definition", "")}')
    try:
        res = parse_json(gemini_json(CANON_PROMPT, "\n".join(lines)))
    except Exception:
        return None
    mid = res.get("match_id")
    if not mid or str(mid).lower() == "none":
        return None
    return next((t for t in shortlist if t["id"] == mid), None)

# Verify ALL of an article's mints in ONE call (CANON_MODE="per_article").
# items: [{"key": str, "label", "definition", "shortlist": [vocab entry, ...]}] -> {key: entry or None}.
def llm_canonicalize_batch(items):
    todo = [it for it in items if it["shortlist"]]           # nothing to match against -> stays a mint
    if not todo:
        return {it["key"]: None for it in items}
    lines = []
    for it in todo:
        lines += [f'NEW TYPE key={it["key"]}', f'  label: {it["label"]}', f'  definition: {it["definition"]}',
                  "  CANDIDATES:"]
        lines += [f'    - id: {t["id"]} | label: {t["canonical_label"]} | definition: {t.get("definition", "")}'
                  for t in it["shortlist"]]
        lines.append("")
    try:
        res = parse_json(gemini_json(CANON_BATCH_PROMPT, "\n".join(lines)))
    except Exception:
        return {it["key"]: None for it in items}
    results = res.get("results", res) if isinstance(res, dict) else res
    by_key  = {str(r.get("key")): r.get("match_id") for r in (results or [])}
    out = {it["key"]: None for it in items}
    for it in todo:
        mid = by_key.get(it["key"])
        if mid and str(mid).lower() != "none":
            out[it["key"]] = next((t for t in it["shortlist"] if t["id"] == mid), None)
    return out

def _schema_ref(label):
    return {"label": label, "source": "schema.org", "vocab_id": None,
            "definition": None, "parent": None, "is_new_to_vocab": False}

def _vocab_ref(ent, is_new=False):
    return {"label": ent["canonical_label"], "source": "vocab", "vocab_id": ent["id"],
            "definition": ent["definition"], "parent": ent["parent"], "is_new_to_vocab": is_new}

# Apply a verifier verdict (matched vocab entry or None) -> merge onto it, or admit a new entry.
def _apply_mint(aid, oie_type, ch, group, vocab, match):
    label = ch.get("label"); definition = ch.get("definition") or ""
    parent = ch.get("parent") or "Thing"
    if match is not None:                                       # merge onto the existing concept
        if label != match["canonical_label"] and label not in match["aliases"]:
            match["aliases"].append(label)
        _bump(match, aid, oie_type, group)
        return _vocab_ref(match)
    if not EXPAND_MODE:                                         # restrict mode: fall back to the parent
        return _schema_ref(parent)
    if not (parent in SCHEMA_LABELS or _is_vocab_label(vocab, parent)):
        parent = "Thing"
    ent = {"id": f"T{vocab['next_id']:04d}", "canonical_label": label, "definition": definition,
           "parent": parent, "aliases": [], "support": 1, "first_seen": aid,
           "provenance": [{"article": aid, "oie_type": oie_type, "mentions": [e["text"] for e in group]}]}
    vocab["types"].append(ent); vocab["next_id"] += 1
    return _vocab_ref(ent, is_new=True)

# per_mint convenience: shortlist -> verify (1 call) -> apply.
def canonicalize_mint(aid, oie_type, ch, group, vocab):
    shortlist = _mint_shortlist(ch.get("label"), ch.get("definition") or "", vocab)
    match     = llm_canonicalize(ch.get("label"), ch.get("definition") or "", shortlist)
    return _apply_mint(aid, oie_type, ch, group, vocab, match)

# Entity-node construction (own dedup, ported from nb3 — every mention lands in exactly one node).
_META_KEYS = ("text", "normalized_value", "confidence")

def _make_node(oie_type, tref, canonical, alias_texts, by_text):
    meta = [by_text[m] for m in ([canonical] + alias_texts) if m in by_text]
    return {
        "canonical_text":         canonical,
        "aliases":                alias_texts,
        "oie_type":               oie_type,
        "chosen_type":            tref.get("label"),
        "type_source":            tref.get("source"),
        "vocab_id":               tref.get("vocab_id"),
        "type_is_new_to_vocab":   tref.get("is_new_to_vocab", False),
        "type_definition":        tref.get("definition"),
        "type_parent":            tref.get("parent"),
        "type_justification":     tref.get("justification"),
        "mention_count":          len(meta),
        "mentions":               [{k: m.get(k) for k in _META_KEYS} for m in meta],
    }

def _apply_clusters(oie_type, entities, tref, clusters):
    """LLM clusters -> canonical nodes; conservative (hallucinated/duplicate members are skipped, so
    every mention lands in exactly one node)."""
    by_text, used, nodes = {e["text"]: e for e in entities}, set(), []
    for cl in clusters or []:
        canonical = cl.get("canonical")
        if canonical not in by_text or canonical in used:
            continue
        aliases = [a for a in (cl.get("aliases") or [])
                   if a in by_text and a != canonical and a not in used]
        if not aliases:
            continue
        for m in [canonical] + aliases:
            used.add(m)
        nodes.append(_make_node(oie_type, tref, canonical, aliases, by_text))
    for e in entities:                                          # remaining mentions -> singletons
        if e["text"] not in used:
            used.add(e["text"])
            nodes.append(_make_node(oie_type, tref, e["text"], [], by_text))
    return nodes

In [49]:
# Process ONE article end-to-end, self-contained (no nb3 needed):
#   retrieve (Hook A) -> ONE LLM call for type-selection + within-group dedup -> canonicalize the mints
#   against the vocabulary (Hook B; per_mint = one call each, per_article = one batched call) -> build
#   canonical entity nodes. Writes typed_deduped_entities_ext.json AND matching_review_ext.json
#   (per OIE type: query text, mentions, the candidates the LLM chose from, and the final chosen type).
def process_article(aid, vocab, write=True, save=True, verbose=True):
    oie, tinfo, types, qtexts, groups = load_article(aid)
    cands  = retrieve_candidates(qtexts, vocab)
    report = llm_select_and_dedup(types, tinfo, groups, cands)

    resolved, pending, n_mint, n_merge = {}, [], 0, 0
    for t in types:
        ch = report[t]["chosen_type"]
        is_mint = ch.get("is_new") or ch.get("source") == "new"
        if is_mint and CANON_MODE == "per_article":                # defer -> one batched verify below
            pending.append((str(len(pending)), t, ch, groups[t],
                            _mint_shortlist(ch.get("label"), ch.get("definition") or "", vocab)))
            continue
        if is_mint:                                                # per_mint: verify now (1 call each)
            before = len(vocab["types"])
            tref = canonicalize_mint(aid, t, ch, groups[t], vocab)
            n_mint += (len(vocab["types"]) > before); n_merge += (len(vocab["types"]) == before)
        else:
            ent = _find_vocab_by_label(vocab, ch.get("label"))     # chose an existing type
            if ent:
                _bump(ent, aid, t, groups[t]); tref = _vocab_ref(ent)
            else:
                tref = _schema_ref(ch.get("label"))
        tref["justification"] = ch.get("justification")
        resolved[t] = tref

    if pending:                                                    # per_article: ONE verify call for all mints
        matches = llm_canonicalize_batch([{"key": k, "label": ch.get("label"),
                                           "definition": ch.get("definition") or "", "shortlist": sl}
                                          for (k, t, ch, grp, sl) in pending])
        for (k, t, ch, grp, sl) in pending:
            before = len(vocab["types"])
            tref = _apply_mint(aid, t, ch, grp, vocab, matches.get(k))
            n_mint += (len(vocab["types"]) > before); n_merge += (len(vocab["types"]) == before)
            tref["justification"] = ch.get("justification")
            resolved[t] = tref

    nodes = []
    for t in types:                                                # build nodes in original type order
        nodes.extend(_apply_clusters(t, groups[t], resolved[t], report[t]["clusters"]))

    if write:
        out_dir = RESULTS_DIR / aid
        (out_dir / "typed_deduped_entities_ext.json").write_text(json.dumps(
            {"article_id": aid, "match_model": MATCH_MODEL, "expand_mode": EXPAND_MODE, "entities": nodes},
            indent=2, ensure_ascii=False))
        # Review file: one row per OIE type with the query text, mentions, the candidates the LLM
        # actually chose from (captured at THIS article's vocab state), the final chosen type, and the
        # LLM's original proposal (so you can see mints that were later canonicalized to an existing type).
        review = [{
            "oie_type":             t,
            "oie_definition":       tinfo.get(t, {}).get("definition"),
            "embedding_query_text": qtexts[i],
            "chosen_type": {
                "label":            resolved[t].get("label"),
                "source":           resolved[t].get("source"),
                "vocab_id":         resolved[t].get("vocab_id"),
                "is_new_to_vocab":  resolved[t].get("is_new_to_vocab", False),
                "definition":       resolved[t].get("definition"),
                "parent":           resolved[t].get("parent"),
                "justification":    resolved[t].get("justification"),
            },
            "llm_proposed":         {k: (report[t]["chosen_type"] or {}).get(k)
                                     for k in ("source", "label", "is_new")},
            "mentions":             [e["text"] for e in groups[t]],
            "candidates":           cands[i],
        } for i, t in enumerate(types)]
        (out_dir / "matching_review_ext.json").write_text(json.dumps(
            {"article_id": aid, "match_model": MATCH_MODEL, "expand_mode": EXPAND_MODE, "types": review},
            indent=2, ensure_ascii=False))
    if save:
        save_vocab(vocab)
    if verbose:
        merged = sum(1 for n in nodes if n["mention_count"] > 1)
        print(f"  {aid}: {len(types)} types, {len(nodes)} nodes ({merged} deduped) "
              f"| +{n_mint} minted, {n_merge} canon-merged | vocab now {len(vocab['types'])}")
    return nodes, vocab

## Dry run — one article, no writes

In [50]:
# # Dry run on the FIRST article against a COPY of the current vocab (no writes, no vocab mutation).
# _vocab_preview = copy.deepcopy(load_vocab())
# _nodes, _ = process_article(ARTICLE_ORDER[0], _vocab_preview, write=False, save=False)
# for (src, lbl), c in Counter((n.get("type_source"), n["chosen_type"]) for n in _nodes).most_common():
#     print(f"  {str(src):<11} {lbl}  x{c}")

## Build — incremental vocabulary over all articles (in order)

In [53]:
# Incremental build. FRESH=True: start from an EMPTY vocab and process ALL of ARTICLE_ORDER (reproducible).
# FRESH=False: keep the existing type_vocabulary.json and process only NEW_ARTICLES (future growth).
FRESH        = True
NEW_ARTICLES = []          # used only when FRESH=False

if FRESH:
    vocab = {"types": [], "next_id": 1}; save_vocab(vocab); todo = ARTICLE_ORDER
else:
    vocab = load_vocab()
    todo  = NEW_ARTICLES or [a for a in ARTICLE_ORDER
                             if not (RESULTS_DIR / a / "typed_deduped_entities_ext.json").exists()]

print(f"building vocabulary over {len(todo)} article(s): {todo}\n")
for aid in todo:
    process_article(aid, vocab, write=True, save=True, verbose=True)
print(f"\ndone — {len(vocab['types'])} canonical types in {VOCAB_PATH}")

building vocabulary over 4 article(s): ['art1', 'art2', 'art4', 'art5']



ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 1.508249609s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'}, 

## Report + verify

In [13]:
# Vocabulary summary, then verify each ext file's entity multiset equals its oie_entities.json input
# (self-contained check — no mention lost or duplicated across canonical nodes).
vocab = load_vocab()
print(f"VOCABULARY — {len(vocab['types'])} canonical types\n" + "-" * 74)
for t in sorted(vocab["types"], key=lambda x: -x.get("support", 0)):
    al = f"  aka {t['aliases']}" if t.get("aliases") else ""
    print(f"  {t['id']}  {t['canonical_label']:<24} <: {str(t.get('parent')):<16} support={t.get('support')}{al}")

print("\nVERIFY ext vs oie_entities (mentions preserved)\n" + "-" * 74)
ok = True
for aid in ARTICLE_ORDER:
    ext = RESULTS_DIR / aid / "typed_deduped_entities_ext.json"
    if not ext.exists():
        print(f"  {aid:<8} (no ext file)"); continue
    oie = json.loads((RESULTS_DIR / aid / "oie_entities.json").read_text())
    x   = json.loads(ext.read_text())["entities"]
    orig = sorted((e["type"], e["text"]) for e in oie["entities"])
    got  = sorted((n["oie_type"], m["text"]) for n in x for m in n["mentions"])
    nvoc = sum(1 for n in x if n.get("type_source") == "vocab")
    merged = sum(1 for n in x if n["mention_count"] > 1)
    ok &= (orig == got)
    print(f"  {aid:<8} {len(oie['entities']):>4} mentions -> {len(x):>4} nodes "
          f"({merged} deduped, {nvoc} via vocab)   {'OK' if orig == got else 'FAIL: multiset changed'}")
print("-" * 74); print("consistent" if ok else "inconsistencies")

VOCABULARY — 58 canonical types
--------------------------------------------------------------------------
  T0005  GovernmentOfficialTitle  <: Role             support=5
  T0041  CalendarYear             <: Duration         support=3
  T0001  DeathCount               <: Quantity         support=2
  T0003  Month                    <: Duration         support=2
  T0004  Percentage               <: Quantity         support=2
  T0006  Temperature              <: Quantity         support=2
  T0011  HolidayWeekend           <: Event            support=2
  T0016  PersonCount              <: Quantity         support=2
  T0021  PolicyStrategy           <: CreativeWork     support=2
  T0035  SocialGroup              <: Audience         support=2
  T0039  FootballPlayer           <: Person           support=2
  T0042  RelativeDate             <: Intangible       support=2
  T0002  WeatherPhenomenon        <: Event            support=1
  T0007  WarningAlert             <: Message          support

# ─────────────────────────── GLOBAL RE-CONSOLIDATION (optional) ───────────────────────────
## Everything below is the optional global-reconsolidation block

Run the pipeline above (through **Build** and **Report + verify**) to produce the raw incremental
vocabulary (`type_vocabulary.json`). The cells below are a **self-contained, optional block** — run
them to clean up the vocabulary globally, or **skip them entirely** to use the raw vocabulary as-is.
Nothing above depends on these cells.


The one caveat is scale. This works only while the whole vocabulary (the extension only) fits comfortably in a single prompt. If the vocab grows into the many hundreds/thousands, you'd have to reintroduce embeddings — either to chunk the list or to shortlist candidate pairs — so the model isn't asked to reason over an unwieldy list in one shot. 


The block, in order:
1. **Config + prompt** — `SUBTYPE_MODE`, the separate output paths, `load_consolidated`, `CONSOLIDATE_ALL_PROMPT`.
2. **`consolidate_vocab`** — ONE Gemini call → writes `type_vocabulary_consolidated.json` (raw untouched).
3. **`refresh_ext_after_consolidation`** — no LLM; reflects merges / re-parentings in the `*_ext.json`.
4. **`build_type_hierarchy`** — indented schema.org + extension tree (works with or without step 2).

In [38]:
# Global consolidation prompt — used by the ONE-call consolidate_vocab below.
# SUBTYPE_MODE controls how a SPECIFIC type is related to a broader type ALREADY in the vocabulary:
#   "off"      : only merge exact synonyms (no broader/narrower handling).
#   "subclass" : KEEP the specific type, set its parent to the broader vocab type — builds an is-a
#                hierarchy; preserves granularity + is-a links.
#   "merge"    : COLLAPSE the specific type into the broader parent (child removed, folded into parent).
SUBTYPE_MODE = "subclass"
assert SUBTYPE_MODE in ("off", "subclass", "merge"), "SUBTYPE_MODE must be 'off', 'subclass', or 'merge'"

# Consolidation writes to SEPARATE files — the raw incremental type_vocabulary.json is left INTACT.
CONSOLIDATED_VOCAB_PATH   = RESULTS_DIR / "type_vocabulary_consolidated.json"   # the consolidated snapshot
CONSOLIDATION_REPORT_PATH = RESULTS_DIR / "type_consolidation_report.json"      # what merged / re-parented
def load_consolidated():
    """The consolidated vocab if it exists, else fall back to the raw incremental one."""
    return json.loads(CONSOLIDATED_VOCAB_PATH.read_text()) if CONSOLIDATED_VOCAB_PATH.exists() else load_vocab()

CONSOLIDATE_ALL_PROMPT = """You are organizing a vocabulary of entity TYPES. You are given the full list of types (id, label, parent, support, definition). Produce GROUPS of two kinds:

SYNONYM — two or more entries that denote the SAME concept (true synonyms: the same kind of thing, differing only in wording or label). Set keep_id to any member and merge_ids to the rest.
SUBTYPE — one or more SPECIFIC (child) types that are all kinds of a single more GENERAL (parent) type WHICH IS ALSO PRESENT in this vocabulary — an is-a relationship. Set keep_id to the GENERAL parent type's id and merge_ids to the specific child ones. Only do this when the parent type already appears in the list (never invent one) and genuinely subsumes the children — every mention under a child type must still be a valid instance of the parent type.

Rules:
- Judge by each type's LABEL and DEFINITION (not the label alone). Be conservative. For SYNONYM, only group true equivalents. For SUBTYPE, only relate a child to a parent when the parent clearly and safely subsumes it; if a real distinction would be lost, leave them separate.
- Each type belongs to at most one group. Every group has a keep_id and one or more merge_ids. Give a one-line reason and set "kind" to "synonym" or "subtype".

Return ONLY valid JSON — no prose, no code fences. Angle-bracket placeholders show the shape, not literal values:
{"groups": [{"kind": "subtype", "keep_id": "<parent id>", "merge_ids": ["<child id>"], "reason": "one short line"}, {"kind": "synonym", "keep_id": "<id>", "merge_ids": ["<id>"], "reason": "one short line"}]}"""
print(f"consolidation prompt loaded | SUBTYPE_MODE={SUBTYPE_MODE!r} | consolidated -> {CONSOLIDATED_VOCAB_PATH.name}")

consolidation prompt loaded | SUBTYPE_MODE='subclass' | consolidated -> type_vocabulary_consolidated.json


In [ ]:
import time

def consolidate_vocab(vocab, write_report=True, verbose=True):
    """Global re-consolidation in ONE Gemini call. The model returns SYNONYM and SUBTYPE groups;
    how SUBTYPE (is-a) groups are applied depends on SUBTYPE_MODE:
      "off"      -> ignore them (only synonyms merge).
      "subclass" -> KEEP each child type, set its parent to the broader vocab type (build hierarchy).
      "merge"    -> fold each child type into the broader parent (removed; label -> alias).
    SYNONYM groups always merge onto the highest-support member. Writes the consolidated vocab to
    CONSOLIDATED_VOCAB_PATH and a report to CONSOLIDATION_REPORT_PATH — the raw type_vocabulary.json is
    NOT modified (pass load_vocab() as `vocab`)."""
    ents  = list(vocab["types"])
    by_id = {t["id"]: t for t in ents}

    def _articles_of(t):
        return [p.get("article") for p in t.get("provenance", []) if p.get("article")]
    articles_before = sorted({a for t in ents for a in _articles_of(t)})

    groups = []
    if len(ents) >= 2:
        lines = [f'- id: {t["id"]} | label: {t["canonical_label"]} | parent: {t.get("parent")} '
                 f'| support: {t.get("support", 0)} | def: {t.get("definition", "")}'
                 + (f' | aka: {", ".join(t["aliases"])}' if t.get("aliases") else "")
                 for t in sorted(ents, key=lambda x: x["id"])]
        gate = ("\nBoth SYNONYM and SUBTYPE groups are enabled." if SUBTYPE_MODE != "off" else
                "\nReturn only SYNONYM groups; do NOT relate broader/narrower (subtype) types.")
        try:
            res = parse_json(gemini_json(CONSOLIDATE_ALL_PROMPT + gate, "VOCABULARY:\n" + "\n".join(lines)))
            groups = (res.get("groups", res) if isinstance(res, dict) else res) or []
        except Exception as ex:
            print(f"[WARN] consolidation call failed ({type(ex).__name__}: {ex})")

    merges, reparents, merged_ids = [], [], set()
    for g in groups:
        kind = (g.get("kind") or "synonym").lower()
        keep = g.get("keep_id")
        ids  = list(dict.fromkeys(i for i in ([keep] + (g.get("merge_ids") or []))
                                  if i in by_id and i not in merged_ids))
        if len(ids) < 2:
            continue
        is_subtype = kind.startswith("subtype")

        # SUBTYPE + subclass: keep the children, just re-parent them onto the broader parent
        if is_subtype and SUBTYPE_MODE == "subclass":
            if keep not in by_id or keep in merged_ids:
                continue
            parent = by_id[keep]
            rec = {"kind": "subtype", "action": "subclass", "parent_id": parent["id"],
                   "parent_label": parent["canonical_label"], "reason": g.get("reason"), "children": []}
            for cid in ids:
                if cid == keep:
                    continue
                child = by_id[cid]
                rec["children"].append({"id": cid, "label": child["canonical_label"],
                                        "old_parent": child.get("parent")})
                child["parent"] = parent["canonical_label"]       # keep the child type; re-parent it
            if rec["children"]:
                reparents.append(rec)
            continue

        # otherwise MERGE: synonyms always; subtype only when SUBTYPE_MODE == "merge"
        if is_subtype and SUBTYPE_MODE != "merge":
            continue
        members = [by_id[i] for i in ids]
        if is_subtype:
            if keep not in by_id or keep in merged_ids:
                continue
            survivor = by_id[keep]                                # broader parent survives
        else:
            survivor = max(members, key=lambda t: (t.get("support", 0), -int(t["id"][1:])))
        rec = {"kind": kind, "action": "merge", "survivor_id": survivor["id"],
               "survivor_label": survivor["canonical_label"], "reason": g.get("reason"), "merged": [],
               "articles": sorted({a for t in members for a in _articles_of(t)})}
        for m in members:
            if m["id"] == survivor["id"]:
                continue
            for lbl in [m["canonical_label"]] + m.get("aliases", []):
                if lbl != survivor["canonical_label"] and lbl not in survivor["aliases"]:
                    survivor["aliases"].append(lbl)
            survivor["support"] = survivor.get("support", 0) + m.get("support", 0)
            survivor.setdefault("provenance", []).extend(m.get("provenance", []))
            rec["merged"].append({"id": m["id"], "label": m["canonical_label"],
                                  "support": m.get("support", 0), "articles": _articles_of(m)})
            merged_ids.add(m["id"])
        if rec["merged"]:
            rec["resulting_support"] = survivor["support"]
            rec["resulting_aliases"] = list(survivor["aliases"])
            merges.append(rec)

    if merged_ids:
        vocab["types"] = [t for t in vocab["types"] if t["id"] not in merged_ids]

    # write the CONSOLIDATED vocabulary to its own file (raw type_vocabulary.json stays untouched)
    CONSOLIDATED_VOCAB_PATH.write_text(json.dumps(vocab, indent=2, ensure_ascii=False))

    n_reparented = sum(len(r["children"]) for r in reparents)
    report = {
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"), "match_model": MATCH_MODEL,
        "gemini_model": GEMINI_MODEL, "canon_mode": CANON_MODE, "expand_mode": EXPAND_MODE,
        "subtype_mode": SUBTYPE_MODE, "source_vocab": str(VOCAB_PATH.name),
        "consolidated_vocab": str(CONSOLIDATED_VOCAB_PATH.name),
        "articles_used": articles_before, "n_articles": len(articles_before),
        "vocab_size_before": len(ents), "vocab_size_after": len(vocab["types"]),
        "n_merges": len(merges), "n_types_removed": len(merged_ids), "n_reparented": n_reparented,
        "merges": merges, "reparents": reparents,
    }
    if write_report:
        CONSOLIDATION_REPORT_PATH.write_text(json.dumps(report, indent=2, ensure_ascii=False))

    if verbose:
        print(f"consolidation (1 call): subtype_mode={SUBTYPE_MODE!r} | {len(merges)} merge-group(s), "
              f"{len(merged_ids)} removed, {n_reparented} re-parented | vocab {len(ents)} -> {len(vocab['types'])}")
        for r in merges:
            print(f"  [merge/{r['kind'][:4]}] keep {r['survivor_id']} {r['survivor_label']!r} <= "
                  f"{[m['label'] for m in r['merged']]}")
        for r in reparents:
            print(f"  [subtype] {[c['label'] for c in r['children']]} <: {r['parent_label']}")
        print(f"  consolidated vocab -> {CONSOLIDATED_VOCAB_PATH}  (raw {VOCAB_PATH.name} untouched)")
        if write_report:
            print(f"  report -> {CONSOLIDATION_REPORT_PATH}")
    return report

# Runs on the RAW incremental vocab and writes the consolidated result to a separate file.
_ = consolidate_vocab(load_vocab())

### Refresh `*_ext.json` after consolidation — no LLM

Apply the consolidation's `merged_id → survivor` mapping to the per-article files deterministically
(remap retired `vocab_id`s to their survivor's id/label/definition/parent). Cheaper and more faithful
than `FRESH=True`: it keeps the original type decisions and only reflects the merges. Run this right
after `consolidate_vocab`.

In [ ]:
# Reflect a consolidation run in the per-article *_ext.json WITHOUT the LLM. Two effects:
#   - MERGED types: nodes pointing at a removed vocab_id are remapped to its survivor.
#   - RE-PARENTED types (subclass mode): the type survives, so we re-sync every vocab-typed node's
#     label / definition / parent from the CONSOLIDATED vocabulary (picks up the new parent).
# Pure bookkeeping — type DECISIONS are untouched. Run right after consolidate_vocab.
def refresh_ext_after_consolidation(report=None, verbose=True):
    if report is None:
        if not CONSOLIDATION_REPORT_PATH.exists():
            print("no consolidation report found — run consolidate_vocab first"); return
        report = json.loads(CONSOLIDATION_REPORT_PATH.read_text())
    remap = {m["id"]: r["survivor_id"] for r in report.get("merges", []) for m in r.get("merged", [])}
    by_id = {t["id"]: t for t in load_consolidated()["types"]}    # survivors + re-parented types
    total = 0
    for fp in sorted(RESULTS_DIR.glob("*/typed_deduped_entities_ext.json")):
        data, changed = json.loads(fp.read_text()), 0
        for n in data["entities"]:
            v = n.get("vocab_id")
            if not v:
                continue
            s = by_id.get(remap.get(v, v))                        # follow a merge, else same id
            if s is None:
                continue
            new = (s["id"], s["canonical_label"], s.get("definition"), s.get("parent"), False)
            old = (n.get("vocab_id"), n.get("chosen_type"), n.get("type_definition"),
                   n.get("type_parent"), n.get("type_is_new_to_vocab"))
            if new != old:
                (n["vocab_id"], n["chosen_type"], n["type_definition"], n["type_parent"],
                 n["type_is_new_to_vocab"]) = new
                changed += 1
        if changed:
            fp.write_text(json.dumps(data, indent=2, ensure_ascii=False)); total += changed
            if verbose:
                print(f"  {fp.parent.name}: updated {changed} node(s)")
    print(f"done — updated {total} node(s) across ext files (no LLM calls)")

refresh_ext_after_consolidation()

## Type hierarchy (schema.org + extension)

One indented tree of the whole type space — schema.org classes and the extension (minted) types
together, each tagged `[schema.org]` or `[extension]`, subclasses nested under their parent. Built
from the **consolidated** vocab (so `subclass` re-parentings appear). Writes `results/type_hierarchy.txt`.
Set `INCLUDE_ALL_SCHEMA=True` to include the entire schema.org tree, or leave it `False` to show just
the branches your extension types hang off.

In [ ]:
# Type hierarchy: schema.org classes + extension (vocab) types in ONE indented tree, each tagged
# [schema.org] or [extension]; children (subclasses) are nested under their parent. Built from the
# CONSOLIDATED vocab so re-parentings show up. Writes results/type_hierarchy.txt.
#   INCLUDE_ALL_SCHEMA=False -> only the schema.org branches that contain an extension type (the
#                              ancestors of your minted types) + all extension types.
#   INCLUDE_ALL_SCHEMA=True  -> the entire schema.org tree as well (large: ~900+ lines).
INCLUDE_ALL_SCHEMA = False
HIERARCHY_PATH = RESULTS_DIR / "type_hierarchy.txt"

def build_type_hierarchy(include_all_schema=INCLUDE_ALL_SCHEMA, write_path=HIERARCHY_PATH, show=50):
    from collections import defaultdict
    vocab = load_consolidated()
    schema_labels = {c["label"] for c in schema_pool}
    schema_parent = {c["label"]: (c["path"].split(" > ")[-1] if c.get("path") else None) for c in schema_pool}
    ext = {t["canonical_label"]: t for t in vocab["types"]}

    parent_of, kind = {}, {}
    for lbl in schema_labels:
        parent_of[lbl] = schema_parent.get(lbl); kind[lbl] = "schema.org"
    for lbl, t in ext.items():                                   # extension overrides if same label
        parent_of[lbl] = t.get("parent"); kind[lbl] = "extension"

    if include_all_schema:
        included = set(schema_labels) | set(ext)
    else:                                                        # only extensions + their ancestor chains
        included = set(ext)
        for lbl in list(ext):
            cur, seen = parent_of.get(lbl), set()
            while cur and cur not in seen and cur not in included:
                seen.add(cur); included.add(cur); cur = parent_of.get(cur)

    children, roots = defaultdict(list), []
    for lbl in included:
        p = parent_of.get(lbl)
        (children[p].append(lbl) if p in included else roots.append(lbl))
    for p in children:
        children[p].sort()

    lines = []
    def walk(lbl, depth, seen):
        tag   = "[extension]" if kind.get(lbl) == "extension" else "[schema.org]"
        extra = f"  (support {ext[lbl].get('support', 0)})" if lbl in ext else ""
        lines.append("  " * depth + f"{lbl}  {tag}{extra}")
        for c in children.get(lbl, []):
            if c not in seen:
                walk(c, depth + 1, seen | {c})
    for r in sorted(set(roots)):
        walk(r, 0, {r})

    text = "\n".join(lines)
    if write_path:
        write_path.write_text(text)
    print(f"type hierarchy: {len(lines)} nodes ({len(ext)} extension) | "
          f"include_all_schema={include_all_schema} -> {write_path}\n" + "-" * 74)
    print(text if len(lines) <= show else "\n".join(lines[:show]) + f"\n  ... ({len(lines) - show} more — see file)")
    return text

_ = build_type_hierarchy()